### Try to understand the dataset 

In [4]:
import json
import sys
from pathlib import Path
from openai import OpenAI, APIError

In [5]:
data_path = Path("../../data/convfinqa_dataset.json")
data = json.load(open(data_path))

In [6]:
# basic information
print("Keys in the dataset:", data.keys())

Keys in the dataset: dict_keys(['train', 'dev'])


In [7]:
train_data = data.get("train", [])
dev_data = data.get("dev", [])
all_data = train_data + dev_data

print("Number of train samples:", len(train_data))
print("Number of dev samples:", len(dev_data))
print("Total number of samples:", len(all_data))

Number of train samples: 3037
Number of dev samples: 421
Total number of samples: 3458


In [8]:
train_type2 = sum(1 for sample in train_data if sample["features"]["has_type2_question"])
print("Number of Type 2 training samples:", train_type2)

dev_type2 = sum(1 for sample in dev_data if sample["features"]["has_type2_question"])
print("Number of Type 2 development samples:", dev_type2)

Number of Type 2 training samples: 889
Number of Type 2 development samples: 121


In [35]:
# find a sample with type 2 question
sample_with_type2 = next((sample for sample in all_data if sample["features"]["has_type2_question"]), None)
if sample_with_type2:
    print("Sample with Type 2 question found:")
    print(json.dumps(sample_with_type2, indent=2))

Sample with Type 2 question found:
{
  "id": "Double_UPS/2009/page_33.pdf",
  "doc": {
    "pre_text": "( 1 ) includes shares repurchased through our publicly announced share repurchase program and shares tendered to pay the exercise price and tax withholding on employee stock options . shareowner return performance graph the following performance graph and related information shall not be deemed 201csoliciting material 201d or to be 201cfiled 201d with the securities and exchange commission , nor shall such information be incorporated by reference into any future filing under the securities act of 1933 or securities exchange act of 1934 , each as amended , except to the extent that the company specifically incorporates such information by reference into such filing . the following graph shows a five-year comparison of cumulative total shareowners 2019 returns for our class b common stock , the s&p 500 index , and the dow jones transportation average . the comparison of the total cumul

In [10]:
# check whether every sample has a non-empty table
empty_table_train = []
non_empty_table_train = []

for idx, record in enumerate(train_data):
    table = record.get("doc", {}).get("table", [])
    if not table:
        empty_table_train.append(idx)
    else:
        non_empty_table_train.append(idx)

print(len(empty_table_train))
print(len(non_empty_table_train))

empty_table_dev = []
non_empty_table_dev = []

for idx, record in enumerate(dev_data):
    table = record.get("doc", {}).get("table", [])
    if not table:
        empty_table_dev.append(idx)
    else:
        non_empty_table_dev.append(idx)

print(len(empty_table_dev))
print(len(non_empty_table_dev))

0
3037
0
421


In [ ]:
# statistics on different operations and data selections in the dataset
from collections import Counter
import re

all_operations = Counter()
for record in all_data:
    programs = record["dialogue"]["turn_program"]
    for program in programs:
        operation_list = re.findall(r"(\w+)\(", program)
        all_operations.update(operation_list)

for operation, count in all_operations.items():
    print(f"Operation: {operation}, Count: {count}")


# check number of single number selection programs
single_number_selection_count = 0
for record in all_data:
    programs = record["dialogue"]["turn_program"]
    for program in programs:
        if program and not any(op in programs for op in ["filter", "sort", "aggregate", "arithmetic", "comparison", "join"]):
            single_number_selection_count += 1

print(f"Single number selection programs: {single_number_selection_count}")

Operation: subtract, Count: 5131
Operation: divide, Count: 4280
Operation: add, Count: 2457
Operation: multiply, Count: 894
Operation: greater, Count: 40
Operation: exp, Count: 4
Single number selection programs: 12594


In [18]:
# the paper mentionens:
# "For the number selection questions depending on previous references, e.g., 'what is that value in the
#  subsequent year?', the model is mostly able to answer. Also, the model is mostly clear on when to discard 
# the previous context and make the transition to new questions."

# try to find some keywords that indicate the model needs to refer to previous context

previous_context_keywords = ["that", "subsequent", "this", "previous", "prior", "earlier", "former", "last", "before", "preceding", "prior"]

questions_with_previous_context = []
for record in all_data:
    for i, q in enumerate(record["dialogue"]["conv_questions"]):
        if i > 0:
            if any(keyword in q.lower() for keyword in previous_context_keywords):
                questions_with_previous_context.append(q)

print(len(questions_with_previous_context))

3221


In [22]:
from collections import Counter
import statistics

# try to get the distribution of the number of turns in the data
turn_counts = [len(rec["dialogue"]["conv_questions"]) for rec in all_data]

print(f"min: {min(turn_counts)}")
print(f"max: {max(turn_counts)}")
print(f"avg: {statistics.mean(turn_counts):.2f}")
print(f"median: {statistics.median(turn_counts)}")

turn_dist = Counter(turn_counts)
print(f"\n distribution:")
for turns in sorted(turn_dist.keys()):
    count = turn_dist[turns]
    pct = 100 * count / len(all_data)
    print(f"  {turns}: {count:4d} ({pct:5.1f}%)")

print("\ntry to estimation the tokens needed")
max_tokens_needed = 500 + (max(turn_counts) * 50)
avg_tokens_needed = 500 + (statistics.mean(turn_counts) * 50)
print(f"max context: ~{max_tokens_needed} tokens")
print(f"avg context: ~{avg_tokens_needed:.0f} tokens")

min: 1
max: 9
avg: 3.64
median: 4.0

 distribution:
  1:    6 (  0.2%)
  2:  858 ( 24.8%)
  3:  751 ( 21.7%)
  4:  940 ( 27.2%)
  5:  649 ( 18.8%)
  6:  184 (  5.3%)
  7:   52 (  1.5%)
  8:   16 (  0.5%)
  9:    2 (  0.1%)

try to estimation the tokens needed
max context: ~950 tokens
avg context: ~682 tokens


In [ ]:
# how important are reference words regarding hybrid dialogues
reference_keywords = ["that", "subsequent", "this", "previous", "prior", "earlier", "former", "last", "before", "preceding", "prior"]

hybrid_with_refs = 0
hybrid_turns_with_refs = 0
hybrid_total_turns = 0

for rec in all_data:
    qa_split = rec["dialogue"]["qa_split"]
    conv_questions = rec["dialogue"]["conv_questions"]
    
    if not any(qa_split):  # is qa_split all false, then not a hybird
        continue
    
    has_ref = False
    for turn_idx, q in enumerate(conv_questions):
        hybrid_total_turns += 1
        q_lower = q.lower()
        
        # check if any contains reference keywords
        if any(kw in q_lower for kw in reference_keywords):
            hybrid_turns_with_refs += 1
            has_ref = True
    
    if has_ref:
        hybrid_with_refs += 1

print(f"all hybrid: {sum(1 for r in all_data if any(r['dialogue']['qa_split']))}")
print(f"hybrid_with_refs: {hybrid_with_refs}")
print(f"  → {100*hybrid_with_refs/sum(1 for r in all_data if any(r['dialogue']['qa_split'])):.1f}%")
print(f"\nhybrid_total_turns {hybrid_total_turns}")
print(f"hybrid_turns_with_refs: {hybrid_turns_with_refs}")
print(f"  → {100*hybrid_turns_with_refs/hybrid_total_turns:.1f}%")

# this could be a good indicator that explicit reference handling could be beneficial

all hybrid: 1010
hybrid_with_refs: 741
  → 73.4%

hybrid_total_turns 3968
hybrid_turns_with_refs: 1521
  → 38.3%


In [1]:
from collections import Counter
from pathlib import Path
import json
import re
import statistics

import pandas as pd

DATA_PATH = Path("../../data/convfinqa_dataset.json")
with DATA_PATH.open() as f:
    dataset = json.load(f)

records = []
turns = []
for split, split_records in dataset.items():
    for record in split_records:
        records.append({
            "split": split,
            "record_id": record["id"],
            "num_turns": len(record["dialogue"]["conv_questions"]),
            "has_type2_question": record["features"].get("has_type2_question", False),
            "is_hybrid": any(record["dialogue"].get("qa_split", [])),
            "has_duplicate_columns": record["features"].get("has_duplicate_columns", False),
            "has_non_numeric_values": record["features"].get("has_non_numeric_values", False),
        })
        for turn_index, question in enumerate(record["dialogue"]["conv_questions"]):
            program = record["dialogue"]["turn_program"][turn_index]
            op_names = re.findall(r"([a-zA-Z_]+)\(", program or "")
            turns.append({
                "split": split,
                "record_id": record["id"],
                "turn_index": turn_index,
                "question": question,
                "program": program,
                "is_number_selection": len(op_names) == 0,
                "is_program_question": len(op_names) > 0,
                "num_program_ops": len(op_names),
                "uses_intermediate_ref": "#" in (program or ""),
                "qa_split": record["dialogue"].get("qa_split", [False] * len(record["dialogue"]["conv_questions"]))[turn_index],
                "executed_answer": record["dialogue"]["executed_answers"][turn_index],
            })

records_df = pd.DataFrame(records)
turns_df = pd.DataFrame(turns)
print(records_df.head())
print(turns_df.head())


   split                       record_id  num_turns  has_type2_question  \
0  train  Single_JKHY/2009/page_28.pdf-3          4               False   
1  train  Single_RSG/2008/page_114.pdf-2          4               False   
2  train  Single_AAPL/2002/page_23.pdf-1          4               False   
3  train   Single_UPS/2009/page_33.pdf-2          6               False   
4  train     Double_UPS/2009/page_33.pdf          7                True   

   is_hybrid  has_duplicate_columns  has_non_numeric_values  
0      False                  False                   False  
1      False                  False                   False  
2      False                  False                   False  
3      False                  False                   False  
4       True                  False                   False  
   split                       record_id  turn_index  \
0  train  Single_JKHY/2009/page_28.pdf-3           0   
1  train  Single_JKHY/2009/page_28.pdf-3           1   
2  train 

In [2]:
# Split-level overview used to justify train/dev usage and dev as held-out validation.
split_summary = (
    records_df.groupby("split")
    .agg(
        records=("record_id", "count"),
        turns=("num_turns", "sum"),
        avg_turns=("num_turns", "mean"),
        type2_records=("has_type2_question", "sum"),
        hybrid_records=("is_hybrid", "sum"),
        duplicate_column_records=("has_duplicate_columns", "sum"),
        non_numeric_table_records=("has_non_numeric_values", "sum"),
    )
)
split_summary["type2_%"] = 100 * split_summary["type2_records"] / split_summary["records"]
split_summary["hybrid_%"] = 100 * split_summary["hybrid_records"] / split_summary["records"]
split_summary.round(2)


,records,turns,avg_turns,type2_records,hybrid_records,duplicate_column_records,non_numeric_table_records,type2_%,hybrid_%
split,,,,,,,,,
dev,421,1490,3.54,121,121,17,32,28.74,28.74
train,3037,11104,3.66,889,889,60,371,29.27,29.27


**Report takeaway:** The local dataset has only `train` and `dev` splits. I use train for development/ablation and reserve dev as held-out validation. About 29% of records are type-2/hybrid conversations, which motivates keeping dialogue history and reporting simple vs. hybrid breakdowns rather than only a single headline score.


In [3]:
# Table-shape analysis. ConvFinQA stores tables as column-oriented JSON dictionaries:
# {column_name: {row_label: value}}. This confirms the task is not OCR/layout extraction.

def table_stats(table):
    columns = list(table.keys())
    row_labels = []
    for column_values in table.values():
        row_labels.extend(column_values.keys())
    row_count = len(dict.fromkeys(row_labels))
    cell_count = sum(len(column_values) for column_values in table.values())
    numeric_cells = sum(
        1
        for column_values in table.values()
        for value in column_values.values()
        if isinstance(value, (int, float))
    )
    return {
        "table_columns": len(columns),
        "table_rows": row_count,
        "table_cells": cell_count,
        "numeric_cells": numeric_cells,
    }

table_rows = []
for split, split_records in dataset.items():
    for record in split_records:
        table_rows.append({"split": split, "record_id": record["id"], **table_stats(record["doc"]["table"])})

tables_df = pd.DataFrame(table_rows)
table_summary = tables_df.groupby("split").agg(
    records=("record_id", "count"),
    median_rows=("table_rows", "median"),
    max_rows=("table_rows", "max"),
    median_columns=("table_columns", "median"),
    max_columns=("table_columns", "max"),
    median_cells=("table_cells", "median"),
    max_cells=("table_cells", "max"),
    median_numeric_cells=("numeric_cells", "median"),
    max_numeric_cells=("numeric_cells", "max"),
)
table_summary


,records,median_rows,max_rows,median_columns,max_columns,median_cells,max_cells,median_numeric_cells,max_numeric_cells
split,,,,,,,,,
dev,421,5.0,14,3.0,8,12.0,48,12.0,48
train,3037,5.0,19,3.0,10,12.0,114,12.0,95


In [4]:
# Show one raw table object and the same table as a row-oriented DataFrame.
example_record = dataset["train"][0]
example_table = example_record["doc"]["table"]
print("Example record:", example_record["id"])
print("Raw table keys / columns:", list(example_table.keys()))

example_df = pd.DataFrame(example_table)
example_df.index.name = "metric"
example_df


Example record: Single_JKHY/2009/page_28.pdf-3
Raw table keys / columns: ['Year ended June 30, 2009', '2008', '2007']


,"Year ended June 30, 2009",2008,2007
metric,,,
net income,103102.0,104222.0,104681.0
non-cash expenses,74397.0,70420.0,56348.0
change in receivables,21214.0,-2913.0,-28853.0
change in deferred revenue,21943.0,5100.0,24576.0
change in other assets and liabilities,-14068.0,4172.0,17495.0
net cash from operating activities,206588.0,181001.0,174247.0


**Report takeaway:** Tables are already pre-extracted in JSON and are column-oriented dictionaries. My system therefore does not solve PDF table extraction; it solves row/value grounding over structured text. This justifies the design choice of converting rows into evidence snippets like `T-31` instead of implementing OCR or raw table reconstruction.


In [5]:
# Question type distribution. This mirrors the paper-style split between number selection and program questions.
question_type_summary = (
    turns_df.groupby("split")
    .agg(
        turns=("question", "count"),
        number_selection=("is_number_selection", "sum"),
        program_questions=("is_program_question", "sum"),
        multi_step_programs=("uses_intermediate_ref", "sum"),
    )
)
question_type_summary["number_selection_%"] = 100 * question_type_summary["number_selection"] / question_type_summary["turns"]
question_type_summary["program_%"] = 100 * question_type_summary["program_questions"] / question_type_summary["turns"]
question_type_summary["multi_step_program_%"] = 100 * question_type_summary["multi_step_programs"] / question_type_summary["turns"]
question_type_summary.round(2)


,turns,number_selection,program_questions,multi_step_programs,number_selection_%,program_%,multi_step_program_%
split,,,,,,,
dev,1490,487,1003,457,32.68,67.32,30.67
train,11104,3911,7193,3247,35.22,64.78,29.24


In [6]:
# Operation distribution in gold programs. This motivates v5's small operation set.
op_counter = Counter()
for program in turns_df["program"]:
    op_counter.update(re.findall(r"([a-zA-Z_]+)\(", program or ""))

op_df = pd.DataFrame(op_counter.most_common(), columns=["operation", "count"])
op_df["% of operation calls"] = 100 * op_df["count"] / op_df["count"].sum()
op_df.round(2)


,operation,count,% of operation calls
0,subtract,5131,40.07
1,divide,4280,33.42
2,add,2457,19.19
3,multiply,894,6.98
4,greater,40,0.31
5,exp,4,0.03


**Report takeaway:** Most non-selection questions require only a compact arithmetic operation set: `subtract`, `divide`, `add`, and `multiply` dominate. This supports v5's lightweight calculation-plan executor: it is intentionally smaller than the paper's full DSL, but covers the operations that matter most for this dataset.


In [7]:
# Conversational dependency indicators. These reference words are not perfect labels,
# but they show why a turn-by-turn chatbot needs previous Q/A history.
reference_pattern = re.compile(
    r"\b(that|this|it|its|their|previous|prior|subsequent|following|same|change|difference|percentage|percent)\b",
    flags=re.IGNORECASE,
)
turns_df["has_reference_word"] = turns_df.apply(
    lambda row: row["turn_index"] > 0 and bool(reference_pattern.search(row["question"])),
    axis=1,
)

reference_summary = turns_df.groupby("split").agg(
    turns=("question", "count"),
    reference_turns=("has_reference_word", "sum"),
    later_turns=("turn_index", lambda s: (s > 0).sum()),
)
reference_summary["reference_%_of_all_turns"] = 100 * reference_summary["reference_turns"] / reference_summary["turns"]
reference_summary["reference_%_of_later_turns"] = 100 * reference_summary["reference_turns"] / reference_summary["later_turns"]
reference_summary.round(2)


,turns,reference_turns,later_turns,reference_%_of_all_turns,reference_%_of_later_turns
split,,,,,
dev,1490,725,1069,48.66,67.82
train,11104,4708,8067,42.40,58.36


In [11]:
# Dialogue length distribution supports turn-level breakdowns.
# Rows are dataset splits; columns are the number of turns in a conversation.
turn_count_distribution = (
    records_df.groupby(["split", "num_turns"])
    .size()
    .rename("records")
    .reset_index()
)
turn_count_distribution.pivot(index="split", columns="num_turns", values="records").fillna(0).astype(int)


num_turns,1,2,3,4,5,6,7,8,9
split,,,,,,,,,
dev,0,116,94,103,88,17,2,1,0
train,6,742,657,837,561,167,50,15,2


**Report takeaway:** Many questions are follow-ups rather than standalone questions, and a large share of later turns contain explicit reference language. This motivates passing conversation history, using clean final-answer history, and reporting accuracy by turn index.


In [9]:
# Evidence-selection motivation: each record contains many possible numbers.
# A full-record prompt asks the model to choose among all of them; evidence selection narrows the focus.
number_pattern = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?%?")

candidate_rows = []
for split, split_records in dataset.items():
    for record in split_records:
        table = record["doc"]["table"]
        table_numeric_cells = table_stats(table)["numeric_cells"]
        text_numbers = len(number_pattern.findall(record["doc"]["pre_text"] + " " + record["doc"]["post_text"]))
        candidate_rows.append({
            "split": split,
            "record_id": record["id"],
            "table_numeric_cells": table_numeric_cells,
            "text_numbers": text_numbers,
            "total_numeric_candidates": table_numeric_cells + text_numbers,
        })

candidate_df = pd.DataFrame(candidate_rows)
candidate_summary = candidate_df.groupby("split").agg(
    median_table_numeric_cells=("table_numeric_cells", "median"),
    median_text_numbers=("text_numbers", "median"),
    median_total_numeric_candidates=("total_numeric_candidates", "median"),
    max_total_numeric_candidates=("total_numeric_candidates", "max"),
)
candidate_summary


,median_table_numeric_cells,median_text_numbers,median_total_numeric_candidates,max_total_numeric_candidates
split,,,,
dev,12.0,36.0,48.0,175
train,12.0,38.0,51.0,276


**Report takeaway:** Even after the dataset gives the relevant record, each record can contain many numeric candidates. This explains why v2/v3/v4/v5 use record-local evidence selection rather than sending the full record and hoping the model picks the right number unaided.


In [10]:
# Version/result table placeholder for the report. These are filled from saved dev evaluations.
# Keeping the result table in the notebook makes the data exploration and method evaluation easy to align.
dev_results = pd.DataFrame([
    {"version": "v1", "method": "full-record baseline", "correct": 987, "total": 1490},
    {"version": "v2", "method": "evidence selection", "correct": 1022, "total": 1490},
    {"version": "v3", "method": "evidence + no-gold verification retry", "correct": 1052, "total": 1490},
    {"version": "v4", "method": "v3 + train-example reasoning retrieval", "correct": 1070, "total": 1490},
    {"version": "v5", "method": "v3 + auditable structured calculation execution", "correct": 1066, "total": 1490},
])
dev_results["accuracy_%"] = 100 * dev_results["correct"] / dev_results["total"]
dev_results.round(2)


,version,method,correct,total,accuracy_%
0,v1,full-record baseline,987,1490,66.24
1,v2,evidence selection,1022,1490,68.59
2,v3,evidence + no-gold verification retry,1052,1490,70.60
3,v4,v3 + train-example reasoning retrieval,1070,1490,71.81
4,v5,v3 + auditable structured calculation execution,1066,1490,71.54


**Report takeaway:** The data exploration motivates the version progression: many numeric candidates motivate evidence selection; messy follow-up reasoning motivates verification and history; arithmetic-heavy program questions motivate v5's structured execution. The final dev table then shows the empirical tradeoff: v4 is the strongest headline accuracy system, while v5 is nearly tied and easier to audit.


### Copyable report facts from this exploration

- The provided JSON contains `3037` train records and `421` dev records, with `11104` train turns and `1490` dev turns.
- About `29%` of records are type-2 / hybrid conversations (`889/3037` train, `121/421` dev), so simple-vs-hybrid and turn-index breakdowns are important.
- Tables are pre-extracted as column-oriented JSON dictionaries, not raw PDF images. Median table size is small (`5` rows, `3` columns, `12` cells), but records can still contain many numeric candidates.
- Dev contains `487` number-selection turns and `1003` program turns. Program questions are the majority (`67.3%` of dev turns), which motivates deterministic calculation execution in v5.
- Gold programs are dominated by a compact arithmetic set: `subtract`, `divide`, `add`, and `multiply`. This supports v5's lightweight executor rather than a full paper-style DSL implementation.
- Later-turn reference language is common: `725/1490` dev turns contain simple reference indicators, and this is `67.8%` of non-initial dev turns. This motivates conversation history and clean final-answer history.
- Evidence selection is justified because the task is not only calculation; each selected record still contains competing table/text numbers, so the system must first ground the right row/value.
